In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os

project_root = os.path.dirname(os.path.abspath(''))
if project_root not in sys.path:
    sys.path.append(project_root)

from FEATURES.features import *
from FEATURES.featuresV2 import *
from PRODUCTION.calculateEVS import *
from PRODUCTION.pipelineV2 import *
from PRODUCTION.teamInfo import teamStarPlayer, projectedStartingFive, mainStartingFive

### Update projected starting lineups

In [2]:
from MODELS.scrapStarting import NBADailyLineups

scraper = NBADailyLineups("https://www.rotowire.com/basketball/nba-lineups.php")
scraper.getDict()  # Scrape the lineups
scraper.updateTeamInfo()  # Update teamInfo.py

Successfully updated /Users/alexgonzalez/Documents/NBA-Prop-Predictor/PRODUCTION/teamInfo.py
Updated 16 teams with confirmed lineups


### Load Model

In [2]:
# Load split NGBoost models (mean, variance, calibration factor, and isotonic calibrator)
pts_mean_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_MEAN_MODEL_PRODUCTIONV2.pkl')
pts_var_model = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_VAR_MODEL_PRODUCTIONV2.pkl')
calibration_factor = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_CALIBRATION_FACTOR_PRODUCTIONV2.pkl')
opt_params = joblib.load('../MODELS/SAVED_MODELS/NGBOOST_PTS_OPT_PARAMS_PRODUCTIONv2.pkl')
model = (pts_mean_model, pts_var_model, calibration_factor, opt_params)  

features = joblib.load('../MODELS/SAVED_MODELS/feature_listV2.pkl')

print(f"Optimized parameters: {opt_params}")
print(f"Loaded models with calibration factor: {calibration_factor}")
print(f"Loaded features: {len(features)}")

Optimized parameters: {'low_score_threshold': np.float64(5.0), 'high_score_threshold': np.float64(12.0), 'low_factor': np.float64(1.1), 'mid_factor': np.float64(1.0), 'high_factor': np.float64(2.5)}
Loaded models with calibration factor: 2.45
Loaded features: 112


### Load Player Data and Bookmaker Data

In [3]:
pd.set_option('display.max_columns', None)
today = datetime.today().strftime('%Y%m%d')  
current_date = datetime.now().strftime('%Y-%m-%d')

s26 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_26.csv').sort_values(by='GAME_DATE')

usData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_US_{today}.csv')
dfsData = pd.read_csv(f'../DATA/CSV_FILES/PROP_DATA/PLAYER_LINES/NBA_DFS_{today}.csv')

dfsData.head()

,BOOKMAKER,CATEGORY,NAME,OVER/UNDER,LINE,ODDS,COMMENCE_TIME,LAST_UPDATE
0,Underdog,player_points,Kevin Durant,Over,25.5,-137,2025-11-30,2025-11-30T13:57:32Z
1,Underdog,player_points,Kevin Durant,Under,25.5,-137,2025-11-30,2025-11-30T13:57:32Z
2,Underdog,player_points,Lauri Markkanen,Over,25.5,-137,2025-11-30,2025-11-30T13:57:32Z
3,Underdog,player_points,Lauri Markkanen,Under,25.5,-137,2025-11-30,2025-11-30T13:57:32Z
4,Underdog,player_points,Keyonte George,Over,20.5,-137,2025-11-30,2025-11-30T13:57:32Z


In [8]:
# Debug: Check what columns are available
print("Columns in s26:", s26.columns.tolist())

Columns in s26: ['Unnamed: 0', 'PLAYER_NAME', 'PLAYER_ID', 'MATCHUP', 'TEAM_ABBREVIATION', 'TEAM_ID', 'OPP_ABBREVIATION', 'HOME_GAME', 'GAME_ID', 'GAME_DATE', 'WL', 'PTS', 'AST', 'REB', 'FGM', 'FGA', 'FG_PCT', 'FG3M', 'FG3A', 'FG3_PCT', 'FTM', 'FTA', 'FT_PCT', 'OREB', 'DREB', 'STL', 'BLK_x', 'TOV', 'PLUS_MINUS', 'FANTASY_PTS', 'POINT_PER_SHOT', 'EFG', 'START_POSITION', 'COMMENT', 'OFF_RATING', 'E_OFF_RATING', 'DEF_RATING', 'E_DEF_RATING', 'NET_RATING', 'OREB_PCT', 'DREB_PCT', 'REB_PCT', 'AST_PCT', 'EFG_PCT', 'AST_TOV', 'USG_PCT', 'TS_PCT', 'E_PACE', 'PACE', 'PIE', 'POSS', 'PACE_PER40', 'E_USG_PCT', 'MIN', 'SPD', 'DIST', 'ORBC', 'DRBC', 'RBC', 'TCHS', 'SAST', 'FTAST', 'PASS', 'CFGM', 'CFGA', 'CFG_PCT', 'UFGM', 'UFGA', 'UFG_PCT', 'DFGM', 'DFGA', 'DFG_PCT', 'percentageFieldGoalsAttempted2pt', 'percentageFieldGoalsAttempted3pt', 'percentagePoints2pt', 'percentagePointsMidrange2pt', 'percentagePoints3pt', 'percentagePointsFastBreak', 'percentagePointsFreeThrow', 'percentagePointsOffTurnover

## Top EVs for 2 leg bets

### Underdog picks

In [4]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points')]

underdogPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


underdogPairs = underdogPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2','ODDS 1', 'ODDS 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']]
underdogPairs.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogPairs.csv', index=False)
underdogPairs.head()

Pre-computing predictions for 92 players...
Processing 88 players...
Generated 3624 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,PROB 1,PROB 2,MODEL SIDE 1,MODEL SIDE 2,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2
628,Steven Adams,Drew Eubanks,6.5,5.5,-115,-106,4.33,8.04,0.565,0.540,under,over,0,-10.31,0.0,Low,Low
2169,Jordan Walsh,Alex Caruso,5.5,5.5,-108,100,8.21,7.80,0.535,0.532,over,over,0,-16.37,0.0,Low,Low
2944,Luguentz Dort,Harrison Barnes,7.5,12.5,100,-110,11.97,19.25,0.525,0.525,over,over,0,-18.96,0.0,High,High
66,Kevin Durant,Jock Landale,25.5,8.5,-110,-110,16.92,12.65,0.524,0.524,under,over,0,-19.23,0.0,High,High
3080,Kris Murray,Anthony Edwards,5.5,28.5,-125,-118,6.89,19.23,0.524,0.523,over,under,0,-19.40,0.0,Low,High


### Prizepicks picks

In [5]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

prizepicksPairs = calculate2LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


pairsPrizepicks = prizepicksPairs[['NAME 1', 'NAME 2', 'LINE 1', 'LINE 2', 'PREDICTION 1', 'PREDICTION 2', 'PROB 1', 'PROB 2', 'MODEL SIDE 1', 'MODEL SIDE 2', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2']].head(10)
pairsPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksPairs.csv', index=False)
prizepicksPairs

Pre-computing predictions for 99 players...
Processing 94 players...
Generated 4130 valid 2-leg combinations


,NAME 1,NAME 2,LINE 1,LINE 2,ODDS 1,ODDS 2,PREDICTION 1,PREDICTION 2,MODEL SIDE 1,MODEL SIDE 2,PROB 1,PROB 2,PROB BOTH,EDGE 1,EDGE 2,COMBINED EDGE,EV%,KELLY FULL,RECOMMENDATION,SIGMA 1,SIGMA 2,SIGMA FLAG 1,SIGMA FLAG 2,CI 1,CI 2,CORRELATION,SAME_GAME,EXPECTED ROI
846,Steven Adams,Drew Eubanks,6.5,5.5,-115,-106,4.33,8.04,under,over,0.565,0.540,0.2990,0.043,0.038,0.036,-10.31,0.0,0,3.14,3.90,Low,Low,"(0.0, 10.5)","(0.4, 15.7)",0.05,0,-10.3
1936,Sam Hauser,Jeremy Sochan,6.0,6.5,-137,105,9.85,11.72,over,over,0.536,0.535,0.2806,-0.028,0.059,0.011,-15.82,0.0,0,5.34,6.08,Med,High,"(0.0, 20.3)","(0.0, 23.6)",0.05,0,-15.8
2479,Jordan Walsh,Alex Caruso,5.5,5.5,-108,100,8.21,7.80,over,over,0.535,0.532,0.2788,0.028,0.044,0.031,-16.37,0.0,0,4.76,4.57,Low,Low,"(0.0, 17.5)","(0.0, 16.7)",0.05,0,-16.4
946,Josh Okogie,Jared McCain,6.5,11.5,110,-105,8.89,7.88,over,under,0.530,0.529,0.2751,0.066,0.030,0.042,-17.48,0.0,0,4.18,5.59,Low,Med,"(0.7, 17.1)","(0.0, 18.8)",0.05,0,-17.5
3529,Luguentz Dort,Harrison Barnes,7.5,12.5,100,-110,11.97,19.25,over,over,0.525,0.525,0.2701,0.037,0.014,0.020,-18.96,0.0,0,6.62,6.44,High,High,"(0.0, 24.9)","(6.6, 31.9)",0.05,0,-19.0
3583,Kris Murray,Jock Landale,5.5,8.5,-125,-110,6.89,12.65,over,over,0.524,0.524,0.2688,-0.018,0.013,-0.009,-19.35,0.0,0,4.18,6.01,Low,High,"(0.0, 15.1)","(0.9, 24.4)",0.05,0,-19.3
1307,Quentin Grimes,Cam Spencer,17.5,9.5,-106,-125,11.95,14.12,under,over,0.521,0.522,0.2665,0.019,-0.020,-0.007,-20.06,0.0,0,6.77,6.54,High,High,"(0.0, 25.2)","(1.3, 26.9)",0.05,0,-20.1
1786,Mouhamed Gueye,Cason Wallace,6.5,8.5,-112,-115,8.27,11.29,over,over,0.520,0.518,0.2642,0.005,-0.004,-0.006,-20.73,0.0,0,4.87,5.82,Low,Med,"(0.0, 17.8)","(0.0, 22.7)",0.05,0,-20.7
867,Isaiah Collier,Luke Kennard,6.5,6.5,-104,-109,7.91,7.97,over,over,0.517,0.517,0.2620,0.020,0.008,0.008,-21.41,0.0,0,4.84,5.07,Low,Med,"(0.0, 17.4)","(0.0, 17.9)",0.05,0,-21.4
2429,Neemias Queta,Dylan Harper,9.5,10.5,-114,-110,12.64,14.07,over,over,0.516,0.517,0.2614,-0.003,0.006,-0.005,-21.57,0.0,0,6.27,6.45,High,High,"(0.3, 24.9)","(1.4, 26.7)",0.05,0,-21.6


## 3 leg parlay

### Underdog picks

In [6]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'Underdog') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

underdogTrios = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)

underdogTrios = underdogTrios[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
underdogTrios.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/underdogTrios.csv', index=False)
underdogTrios.head()

Pre-computing predictions for 82 players...
Processing 78 players...
Generated 64070 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
13126,Steven Adams,Jordan Walsh,Drew Eubanks,6.5,5.5,5.5,4.33,8.21,8.04,0.565,0.535,0.540,under,over,over,0,-11.92,0.0,Low,Low,Low
58566,Alex Caruso,Harrison Barnes,Jock Landale,5.5,12.5,8.5,7.80,19.25,12.65,0.532,0.525,0.524,over,over,over,0,-21.01,0.0,Low,High,High
39045,Mouhamed Gueye,Luguentz Dort,Cam Spencer,6.5,7.5,9.5,8.27,11.97,14.12,0.520,0.525,0.522,over,over,over,0,-22.94,0.0,Low,High,High
33988,Nickeil Alexander-Walker,Brandon Ingram,Kris Murray,19.5,23.5,5.5,14.33,17.18,6.89,0.519,0.518,0.524,under,under,over,0,-24.09,0.0,High,High,Low
17104,Isaiah Collier,Onyeka Okongwu,Dylan Harper,6.5,19.5,10.5,7.91,14.65,14.07,0.517,0.518,0.517,over,under,over,0,-25.25,0.0,Low,Med,High


### Prizepicks picks

In [7]:
dfsPTS = dfsData[(dfsData['BOOKMAKER'] == 'PrizePicks') & (dfsData['CATEGORY'] == 'player_points') & (dfsData['LINE'] < 24) & (dfsData['LINE'] > 5)]

triosPrizepicks = calculate3LegBets(s26, dfsPTS, model, features, current_date, 
                           edge_threshold=4, stake=10, max_player_appearances=1, top_n=10)


triosPrizepicks = triosPrizepicks[['NAME 1', 'NAME 2', 'NAME 3', 'LINE 1', 'LINE 2', 'LINE 3', 'PREDICTION 1', 'PREDICTION 2', 'PREDICTION 3', 'PROB 1', 'PROB 2', 'PROB 3', 'MODEL SIDE 1', 'MODEL SIDE 2', 'MODEL SIDE 3', 'RECOMMENDATION','EV%', 'KELLY FULL', 'SIGMA FLAG 1', 'SIGMA FLAG 2', 'SIGMA FLAG 3']].head(10)
triosPrizepicks.to_csv(f'../DATA/CSV_FILES/PROP_DATA/PROPS_EV/prizepicksTrios.csv', index=False)
triosPrizepicks.head()

Pre-computing predictions for 99 players...
Processing 94 players...
Generated 112578 valid 3-leg combinations


,NAME 1,NAME 2,NAME 3,LINE 1,LINE 2,LINE 3,PREDICTION 1,PREDICTION 2,PREDICTION 3,PROB 1,PROB 2,PROB 3,MODEL SIDE 1,MODEL SIDE 2,MODEL SIDE 3,RECOMMENDATION,EV%,KELLY FULL,SIGMA FLAG 1,SIGMA FLAG 2,SIGMA FLAG 3
31790,Steven Adams,Sam Hauser,Drew Eubanks,6.5,6.0,5.5,4.33,9.85,8.04,0.565,0.536,0.540,under,over,over,0,-11.77,0.0,Low,Med,Low
86348,Jordan Walsh,Alex Caruso,Jeremy Sochan,5.5,5.5,6.5,8.21,7.80,11.72,0.535,0.532,0.535,over,over,over,0,-17.88,0.0,Low,Low,High
37787,Josh Okogie,Jared McCain,Luguentz Dort,6.5,11.5,7.5,8.89,7.88,11.97,0.530,0.529,0.525,over,under,over,0,-20.40,0.0,Low,Med,High
108002,Kris Murray,Harrison Barnes,Jock Landale,5.5,12.5,8.5,6.89,19.25,12.65,0.524,0.525,0.524,over,over,over,0,-22.25,0.0,Low,High,High
49922,Quentin Grimes,Cason Wallace,Cam Spencer,17.5,8.5,9.5,11.95,11.29,14.12,0.521,0.518,0.522,under,over,over,0,-23.95,0.0,High,Med,High


In [8]:
player_name = 'Shai Gilgeous-Alexander'

df = playerScoring(player_name, s26, current_date, teamStarPlayer, projectedStartingFive)
team_df = teamContext(player_name, s26)
opp_df = playerVsOpp(player_name, s26, current_date)
player = playerContext(player_name, s26, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)
matchup = playerMatchup(player_name, s26, current_date)
pred = makePrediction(player_name, s26, model, features, current_date, projectedStartingFive, mainStartingFive, teamStarPlayer)
# len(df)

In [9]:
print(f"playerContext: {player}")
print(f"playerScoring: {df}")
print(f"playerMatchup: {matchup}")
print(f"teamContext: {team_df}")
print(f"playerVsOpp: {opp_df}")
print(f"playerContext: {player}")
print(f"{len(df) + len(team_df) + len(opp_df) + len(player) + len(matchup)}")
print(f"makePrediction: {pred}")

playerContext: [1, 2, 1, 2, 20]
playerScoring: [33.37, 32.85, 19.9, 5.2, 10.05, 0.56, 0.46, 0.91, 0.32, 0.69, 70.15, 14.0, 1.08, 1.02, 1.0, 1.01, 0.97, 0.96, 0.95, 0.88, 0.91, 1.29, 1.14, 1.07, 0.61, 0.69, 0.9, 1.06, 1.07, 1.02, 1.31, 1.34, 1.18, 0.94, 0.95, 0.96, 0.98, 0.94, 0.95, 1.04, 1.08, 1.03, 1.02, 0.98, 0.97, 1.0, 1.25, 1.43, 0.74, 0.81, 1.13, 0.65, 0.75, 0.86, 0.91, 0.68, 0.81, 0.76, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, -0.0, 0, 3.28, 3.38, 3.58, 4.97, 3.61, 4.16, 1.1, 2.16, 3.81, 3.77, 0.02, 0.03, 0.1, 0.1]
playerMatchup: [-0.42, -0.35, -0.5, 0.1, -0.45, -0.01, 0.02, 3.15, 2.1, 0.01, 0.0]
teamContext: [2.86, 3.01, 4.79, 1.16, 1.73, 2.43]
playerVsOpp: [1.75, -3.68, -0.63, -12.56, 2.28, -0.47, np.float64(-1.22), np.float64(0.0), np.float64(-0.0)]
playerContext: [1, 2, 1, 2, 20]
112
makePrediction: 29.609
